In [ ]:
# ============================================================
# Webis Clickbait Corpus 2017
# Dataset Preparation for Clickbait Detection Project
# ------------------------------------------------------------
# Dataset A:
#   postText → truthMean
#
# Dataset C:
#   postText + targetTitle + targetDescription + intro → truthMean
# ============================================================

import json
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# 1. File paths
# ------------------------------------------------------------

instances_path = "/content/instances.jsonl"
truth_path = "/content/truth.jsonl"



In [ ]:

# ------------------------------------------------------------
# 2. Function to safely load JSONL file
# ------------------------------------------------------------

def load_jsonl(file_path):
    """
    Loads a JSONL file line by line.

    Each line in a JSONL file is a separate JSON object.
    This function is safer than pd.read_json(..., lines=True)
    because it can handle problematic lines more gracefully.
    """
    records = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Skipping problematic line {line_number}: {e}")

    return pd.DataFrame(records)

In [ ]:
# ------------------------------------------------------------
# 3. Load instances and truth files
# ------------------------------------------------------------

instances = load_jsonl(instances_path)
truth = load_jsonl(truth_path)

print("Instances shape:", instances.shape)
print("Truth shape:", truth.shape)

print("\nInstances columns:")
print(instances.columns)

print("\nTruth columns:")
print(truth.columns)

#instances.info()

Instances shape: (19538, 9)
Truth shape: (19538, 6)

Instances columns:
Index(['postMedia', 'postText', 'id', 'targetCaptions', 'targetParagraphs',
       'targetTitle', 'postTimestamp', 'targetKeywords', 'targetDescription'],
      dtype='object')

Truth columns:
Index(['truthJudgments', 'truthMean', 'id', 'truthClass', 'truthMedian',
       'truthMode'],
      dtype='object')


In [ ]:
# ------------------------------------------------------------
# 4. Merge both files using the common id column
# ------------------------------------------------------------

df = instances.merge(truth, on="id", how="inner")

print("\nMerged dataset shape:", df.shape)
print(df.head())



Merged dataset shape: (19538, 14)
  postMedia                                           postText  \
0        []  [UK’s response to modern slavery leaving victi...   
1        []                                     [this is good]   
2        []  [The "forgotten" Trump roast: Relive his bruta...   
3        []             [Meet the happiest #dog in the world!]   
4        []  [Tokyo's subway is shut down amid fears over a...   

                   id                                     targetCaptions  \
0  858462320779026433                           [modern-slavery-rex.jpg]   
1  858421020331560960  [In this July 1, 2010 file photo, Dr. Charmain...   
2  858368123753435136  [President Trump will not attend this year's W...   
3  858323428260139008                    [Maru , Maru, Maru, Maru, Maru]   
4  858283602626347008  [All nine lines of Tokyo's subway system were ...   

                                    targetParagraphs  \
0  [Thousands of modern slavery victims have not ...   

In [ ]:
# ------------------------------------------------------------
# 5. Helper functions for cleaning fields
# ------------------------------------------------------------

def list_to_text(value):
    """
    Converts a list field into a plain string.

    In Webis dataset, fields like postText and targetParagraphs
    are often stored as lists.
    """
    if isinstance(value, list):
        return " ".join([str(item).strip() for item in value if str(item).strip()])
    elif pd.isna(value):
        return ""
    else:
        return str(value).strip()


def first_n_paragraphs(value, n=3):
    """
    Extracts the first n paragraphs from targetParagraphs.
    """
    if isinstance(value, list):
        selected = value[:n]
        return " ".join([str(p).strip() for p in selected if str(p).strip()])
    elif pd.isna(value):
        return ""
    else:
        return str(value).strip()


def clean_text(text):
    """
    Performs basic cleaning.
    """
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = " ".join(text.split())

    return text.strip()

In [ ]:
# ------------------------------------------------------------
# 6. Extract required fields
# ------------------------------------------------------------

df["headline"] = df["postText"].apply(list_to_text).apply(clean_text)

df["title"] = df["targetTitle"].apply(clean_text)

df["description"] = df["targetDescription"].apply(clean_text)

df["intro"] = df["targetParagraphs"].apply(
    lambda x: first_n_paragraphs(x, n=3)
).apply(clean_text)

df["score"] = df["truthMean"]

df["class_label"] = df["truthClass"]

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19538 entries, 0 to 19537
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   postMedia          19538 non-null  object 
 1   postText           19538 non-null  object 
 2   id                 19538 non-null  object 
 3   targetCaptions     19538 non-null  object 
 4   targetParagraphs   19538 non-null  object 
 5   targetTitle        19538 non-null  object 
 6   postTimestamp      19538 non-null  object 
 7   targetKeywords     19538 non-null  object 
 8   targetDescription  19538 non-null  object 
 9   truthJudgments     19538 non-null  object 
 10  truthMean          19538 non-null  float64
 11  truthClass         19538 non-null  object 
 12  truthMedian        19538 non-null  float64
 13  truthMode          19538 non-null  float64
 14  headline           19538 non-null  object 
 15  title              19538 non-null  object 
 16  description        195

In [ ]:
# ------------------------------------------------------------
# 7. Remove empty or unusable records
# ------------------------------------------------------------

df_project = df[
    (df["headline"].str.len() > 0) &
    (df["score"].notna())
].copy()

print("\nProject dataset shape after cleaning:", df_project.shape)



Project dataset shape after cleaning: (19484, 20)


In [ ]:
# ------------------------------------------------------------
# 8. Create Dataset A
# ------------------------------------------------------------
# Dataset A:
# Input  : headline/postText only
# Output : truthMean

dataset_A = pd.DataFrame({
    "input_text": df_project["headline"],
    "score": df_project["score"],
    "class_label": df_project["class_label"]
})

print("\nDataset A sample:")
display(dataset_A.head())



Dataset A sample:


,input_text,score,class_label
0,UK’s response to modern slavery leaving victim...,0.133333,no-clickbait
1,this is good,1.000000,clickbait
2,"The ""forgotten"" Trump roast: Relive his brutal...",0.466667,no-clickbait
3,Meet the happiest #dog in the world!,0.933333,clickbait
4,Tokyo's subway is shut down amid fears over an...,0.000000,no-clickbait


In [ ]:
# ------------------------------------------------------------
# 9. Create Dataset C
# ------------------------------------------------------------
# Dataset C:
# Input  : headline + targetTitle + targetDescription + intro
# Output : truthMean

dataset_C = pd.DataFrame({
    "input_text": (
        "HEADLINE: " + df_project["headline"] +
        " TITLE: " + df_project["title"] +
        " DESCRIPTION: " + df_project["description"] +
        " INTRO: " + df_project["intro"]
    ),
    "headline": df_project["headline"],
    "title": df_project["title"],
    "description": df_project["description"],
    "intro": df_project["intro"],
    "score": df_project["score"],
    "class_label": df_project["class_label"]
})

print("\nDataset C sample:")
display(dataset_C.head())



Dataset C sample:


,input_text,headline,title,description,intro,score,class_label
0,HEADLINE: UK’s response to modern slavery leav...,UK’s response to modern slavery leaving victim...,‘Inexcusable’ failures in UK’s response to mod...,“Inexcusable” failures in the UK’s system for ...,Thousands of modern slavery victims have not c...,0.133333,no-clickbait
1,HEADLINE: this is good TITLE: Donald Trump App...,this is good,Donald Trump Appoints Pro-Life Advocate as Ass...,President Donald Trump has appointed pro-life ...,President Donald Trump has appointed the pro-l...,1.000000,clickbait
2,"HEADLINE: The ""forgotten"" Trump roast: Relive ...","The ""forgotten"" Trump roast: Relive his brutal...",The ‘forgotten’ Trump roast: Relive his brutal...,President Trump won't be at this year's White ...,When the White House correspondents’ dinner is...,0.466667,no-clickbait
3,HEADLINE: Meet the happiest #dog in the world!...,Meet the happiest #dog in the world!,"Meet The Happiest Dog In The World, Maru The H...","The article is about Maru, a husky dog who has...",Adorable is probably an understatement. This a...,0.933333,clickbait
4,HEADLINE: Tokyo's subway is shut down amid fea...,Tokyo's subway is shut down amid fears over an...,Tokyo's subway is shut down amid fears over an...,"The temporary suspension, which lasted ten min...",One of Tokyo's major subways systems says it s...,0.000000,no-clickbait


In [ ]:
# ------------------------------------------------------------
# 10. Optional: Convert score into categorical labels
# ------------------------------------------------------------
# This is useful if you want classification instead of regression.

def score_to_category(score):
    """
    Converts truthMean score into three categories.
    """
    if score < 0.33:
        return "low_clickbait"
    elif score < 0.66:
        return "moderate_clickbait"
    else:
        return "high_clickbait"

dataset_A["score_category"] = dataset_A["score"].apply(score_to_category)
dataset_C["score_category"] = dataset_C["score"].apply(score_to_category)

print("\nDataset A category distribution:")
print(dataset_A["score_category"].value_counts())

print("\nDataset C category distribution:")
print(dataset_C["score_category"].value_counts())


Dataset A category distribution:
score_category
low_clickbait         10840
moderate_clickbait     5777
high_clickbait         2867
Name: count, dtype: int64

Dataset C category distribution:
score_category
low_clickbait         10840
moderate_clickbait     5777
high_clickbait         2867
Name: count, dtype: int64


In [ ]:
# ------------------------------------------------------------
# 11. Train-validation-test split
# ------------------------------------------------------------
# 70% train, 15% validation, 15% test

def split_dataset(dataframe, label_column="score_category"):
    """
    Splits dataset into train, validation, and test sets.
    Stratification is done using score_category.
    """

    train_df, temp_df = train_test_split(
        dataframe,
        test_size=0.30,
        random_state=42,
        stratify=dataframe[label_column]
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=42,
        stratify=temp_df[label_column]
    )

    return train_df, val_df, test_df


A_train, A_val, A_test = split_dataset(dataset_A)
C_train, C_val, C_test = split_dataset(dataset_C)

print("\nDataset A splits:")
print("Train:", A_train.shape)
print("Validation:", A_val.shape)
print("Test:", A_test.shape)

print("\nDataset C splits:")
print("Train:", C_train.shape)
print("Validation:", C_val.shape)
print("Test:", C_test.shape)


Dataset A splits:
Train: (13638, 4)
Validation: (2923, 4)
Test: (2923, 4)

Dataset C splits:
Train: (13638, 8)
Validation: (2923, 8)
Test: (2923, 8)


In [ ]:
# ------------------------------------------------------------
# 12. Save datasets as CSV files
# ------------------------------------------------------------

output_dir = "/content/processed_clickbait_datasets"
os.makedirs(output_dir, exist_ok=True)

dataset_A.to_csv(f"{output_dir}/dataset_A_full.csv", index=False)
dataset_C.to_csv(f"{output_dir}/dataset_C_full.csv", index=False)

A_train.to_csv(f"{output_dir}/dataset_A_train.csv", index=False)
A_val.to_csv(f"{output_dir}/dataset_A_val.csv", index=False)
A_test.to_csv(f"{output_dir}/dataset_A_test.csv", index=False)

C_train.to_csv(f"{output_dir}/dataset_C_train.csv", index=False)
C_val.to_csv(f"{output_dir}/dataset_C_val.csv", index=False)
C_test.to_csv(f"{output_dir}/dataset_C_test.csv", index=False)

print("\nAll datasets saved successfully in:")
print(output_dir)


All datasets saved successfully in:
/content/processed_clickbait_datasets


In [ ]:
# Install required libraries
!pip install -q transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [ ]:
# Import Libraries
import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [ ]:
# Choose MiniLM Model

MODEL_NAME = "microsoft/MiniLM-L12-H384-uncased"

In [ ]:
# Function to Train One Dataset

def train_minilm_classifier(
    train_csv,
    val_csv,
    test_csv,
    output_dir,
    max_length=256,
    num_train_epochs=3,
    batch_size=16,
    learning_rate=2e-5
):
    """
    Fine-tunes MiniLM for clickbait classification.

    Required CSV columns:
        input_text
        score_category

    score_category values:
        low_clickbait
        moderate_clickbait
        high_clickbait
    """

    # ------------------------------------------------------------
    # 1. Load CSV files
    # ------------------------------------------------------------
    train_df = pd.read_csv(train_csv)
    val_df = pd.read_csv(val_csv)
    test_df = pd.read_csv(test_csv)

    print("Train shape:", train_df.shape)
    print("Validation shape:", val_df.shape)
    print("Test shape:", test_df.shape)

    print("\nTraining label distribution:")
    print(train_df["score_category"].value_counts())

    # ------------------------------------------------------------
    # 2. Define label mapping
    # ------------------------------------------------------------
    label2id = {
        "low_clickbait": 0,
        "moderate_clickbait": 1,
        "high_clickbait": 2
    }

    id2label = {
        0: "low_clickbait",
        1: "moderate_clickbait",
        2: "high_clickbait"
    }

    train_df["label"] = train_df["score_category"].map(label2id)
    val_df["label"] = val_df["score_category"].map(label2id)
    test_df["label"] = test_df["score_category"].map(label2id)

    # Remove missing rows if any
    train_df = train_df.dropna(subset=["input_text", "label"])
    val_df = val_df.dropna(subset=["input_text", "label"])
    test_df = test_df.dropna(subset=["input_text", "label"])

    train_df["label"] = train_df["label"].astype(int)
    val_df["label"] = val_df["label"].astype(int)
    test_df["label"] = test_df["label"].astype(int)

    # ------------------------------------------------------------
    # 3. Convert pandas DataFrame to Hugging Face Dataset
    # ------------------------------------------------------------
    train_dataset = Dataset.from_pandas(train_df[["input_text", "label"]])
    val_dataset = Dataset.from_pandas(val_df[["input_text", "label"]])
    test_dataset = Dataset.from_pandas(test_df[["input_text", "label"]])

    # ------------------------------------------------------------
    # 4. Load tokenizer
    # ------------------------------------------------------------
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize_function(batch):
        return tokenizer(
            batch["input_text"],
            truncation=True,
            max_length=max_length
        )

    train_dataset = train_dataset.map(tokenize_function, batched=True)
    val_dataset = val_dataset.map(tokenize_function, batched=True)
    test_dataset = test_dataset.map(tokenize_function, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # ------------------------------------------------------------
    # 5. Load MiniLM classification model
    # ------------------------------------------------------------
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3,
        id2label=id2label,
        label2id=label2id
    )

    # ------------------------------------------------------------
    # 6. Define evaluation metrics
    # ------------------------------------------------------------
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)

        accuracy = accuracy_score(labels, predictions)

        precision, recall, f1, _ = precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0
        )

        return {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }

    # ------------------------------------------------------------
    # 7. Training arguments
    # ------------------------------------------------------------
    training_args = TrainingArguments(
    output_dir=output_dir,

    # Evaluate after every epoch
    eval_strategy="epoch",

    # Save a checkpoint after every epoch
    save_strategy="epoch",

    learning_rate=learning_rate,

    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,

    num_train_epochs=num_train_epochs,
    weight_decay=0.01,

    # Display training logs every 50 update steps
    logging_strategy="steps",
    logging_steps=50,

    # Restore the checkpoint having the highest validation F1
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    # Do not connect to WandB, TensorBoard, etc.
    report_to="none"
    )


    # ------------------------------------------------------------
    # 8. Create Trainer
    # ------------------------------------------------------------
    trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    # Current replacement for the old tokenizer= argument
    processing_class=tokenizer,

    data_collator=data_collator,
    compute_metrics=compute_metrics
    )

    # ------------------------------------------------------------
    # 9. Train
    # ------------------------------------------------------------
    trainer.train()

    # ------------------------------------------------------------
    # 10. Evaluate on test set
    # ------------------------------------------------------------
    test_results = trainer.evaluate(test_dataset)
    print("\nTest results:")
    print(test_results)

    # ------------------------------------------------------------
    # 11. Detailed classification report
    # ------------------------------------------------------------
    predictions_output = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions_output.predictions, axis=-1)
    y_true = predictions_output.label_ids

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=[
                "low_clickbait",
                "moderate_clickbait",
                "high_clickbait"
            ],
            zero_division=0
        )
    )

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    # ------------------------------------------------------------
    # 12. Save final model and tokenizer
    # ------------------------------------------------------------
    final_model_dir = os.path.join(output_dir, "final_model")
    trainer.save_model(final_model_dir)
    tokenizer.save_pretrained(final_model_dir)

    print("\nModel saved at:")
    print(final_model_dir)

    return trainer, tokenizer, final_model_dir

In [ ]:
# Train Model 1: Dataset A
# This model is for: before-click mode- headline/post-text only

A_train_csv = "/content/processed_clickbait_datasets/dataset_A_train.csv"
A_val_csv   = "/content/processed_clickbait_datasets/dataset_A_val.csv"
A_test_csv  = "/content/processed_clickbait_datasets/dataset_A_test.csv"

trainer_A, tokenizer_A, model_A_path = train_minilm_classifier(
    train_csv=A_train_csv,
    val_csv=A_val_csv,
    test_csv=A_test_csv,
    output_dir="/content/minilm_clickbait_model_A",
    max_length=128,
    num_train_epochs=3,
    batch_size=16,
    learning_rate=2e-5
)

Train shape: (13638, 4)
Validation shape: (2923, 4)
Test shape: (2923, 4)

Training label distribution:
score_category
low_clickbait         7587
moderate_clickbait    4044
high_clickbait        2007
Name: count, dtype: int64


Map:   0%|          | 0/13638 [00:00<?, ? examples/s]

Map:   0%|          | 0/2923 [00:00<?, ? examples/s]

Map:   0%|          | 0/2923 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.519921,0.653506,3,0.723230,0.718674,0.723230,0.719906



Test results:
{'eval_loss': 0.6535061597824097, 'eval_accuracy': 0.7232295586725966, 'eval_precision': 0.7186738424604857, 'eval_recall': 0.7232295586725966, 'eval_f1': 0.7199058740217908}



Classification Report:
                    precision    recall  f1-score   support

     low_clickbait       0.81      0.86      0.83      1627
moderate_clickbait       0.55      0.54      0.54       866
    high_clickbait       0.70      0.59      0.64       430

          accuracy                           0.72      2923
         macro avg       0.69      0.66      0.67      2923
      weighted avg       0.72      0.72      0.72      2923


Confusion Matrix:
[[1394  219   14]
 [ 307  465   94]
 [  16  159  255]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved at:
/content/minilm_clickbait_model_A/final_model


In [ ]:
# Train Model 2: Dataset C
# after-click mode: headline + title + description + article intro

C_train_csv = "/content/processed_clickbait_datasets/dataset_C_train.csv"
C_val_csv   = "/content/processed_clickbait_datasets/dataset_C_val.csv"
C_test_csv  = "/content/processed_clickbait_datasets/dataset_C_test.csv"

trainer_C, tokenizer_C, model_C_path = train_minilm_classifier(
    train_csv=C_train_csv,
    val_csv=C_val_csv,
    test_csv=C_test_csv,
    output_dir="/content/minilm_clickbait_model_C",
    max_length=512,
    num_train_epochs=3,
    batch_size=16,
    learning_rate=2e-5
)

Train shape: (13638, 8)
Validation shape: (2923, 8)
Test shape: (2923, 8)

Training label distribution:
score_category
low_clickbait         7587
moderate_clickbait    4044
high_clickbait        2007
Name: count, dtype: int64


Map:   0%|          | 0/13638 [00:00<?, ? examples/s]

Map:   0%|          | 0/2923 [00:00<?, ? examples/s]

Map:   0%|          | 0/2923 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.530289,0.650477,3,0.715361,0.709153,0.715361,0.711696



Test results:
{'eval_loss': 0.6504771113395691, 'eval_accuracy': 0.715360930550804, 'eval_precision': 0.7091531741485897, 'eval_recall': 0.715360930550804, 'eval_f1': 0.711695764810848}



Classification Report:
                    precision    recall  f1-score   support

     low_clickbait       0.81      0.85      0.83      1627
moderate_clickbait       0.54      0.51      0.52       866
    high_clickbait       0.68      0.64      0.66       430

          accuracy                           0.72      2923
         macro avg       0.68      0.66      0.67      2923
      weighted avg       0.71      0.72      0.71      2923


Confusion Matrix:
[[1379  235   13]
 [ 313  438  115]
 [  18  138  274]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved at:
/content/minilm_clickbait_model_C/final_model


In [ ]:
# Distribution of token-size across samples of the data
# To determine the max tokens to be considered for training

import numpy as np
from transformers import AutoTokenizer

# Initialize the tokenizer globally
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def calculate_token_length(text):
    encoded = tokenizer(
        str(text),
        add_special_tokens=True,
        truncation=False
    )

    return len(encoded["input_ids"])


token_lengths = dataset_C["input_text"].apply(
    calculate_token_length
)

print("Number of samples:", len(token_lengths))
print("Minimum:", token_lengths.min())
print("Mean:", token_lengths.mean())
print("Median:", token_lengths.median())
print("Maximum:", token_lengths.max())

print("\nPercentiles:")

for percentile in [50, 75, 90, 95, 99]:
    value = np.percentile(
        token_lengths,
        percentile
    )

    print(
        f"{percentile}th percentile: "
        f"{value:.0f} tokens"
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Number of samples: 19484
Minimum: 30
Mean: 197.61080886881544
Median: 185.0
Maximum: 25651

Percentiles:
50th percentile: 185 tokens
75th percentile: 229 tokens
90th percentile: 283 tokens
95th percentile: 327 tokens
99th percentile: 469 tokens


In [ ]:
if "train_df" in globals():
    print("train_df exists.")
else:
    print("train_df has not been defined.")

train_df has not been defined.
